# 私募合规语料全量官网复核

快照日期：2026-07-16。数据真源为 `D:\FUND_COMPLIANCE\CSRC`；本笔记本仅读取审计产物，不修改代码、raw、canonical 或 reports。

## tl;dr

- 私募合规毛池 862 条，剔除 37 条词面误命中后，最终全集 825 条（规则 146，参考/执法 679）。
- 1,008/1,008 个现有官方来源可访问；协会官网独立清单的 598 条私募候选中，597 条与本地 URL 和标准化标题完全对应，1 条规则页漏入库。
- 291/291 个本地 PDF 可解析且无哈希漂移；358/358 个在线 PDF 链接具有有效 PDF 签名。
- 主要风险不是普遍标题错配，而是效力/时效错误、同 URL 重复入库、扫描件不可检索和 5 个空附件端点。

In [1]:
from pathlib import Path
import csv, json
from collections import Counter
from IPython.display import Markdown, display

AUDIT_DIR = Path('/home/anjie/projects/csrc-law-crawler/tmp/audits/private_compliance_full_20260716')
OUT = AUDIT_DIR / "output"

def read_csv(name):
    with (OUT / name).open(encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

def read_json(name):
    return json.loads((OUT / name).read_text(encoding="utf-8"))

def read_jsonl(name):
    return [json.loads(line) for line in (OUT / name).read_text(encoding="utf-8").splitlines()]

inventory = read_csv("candidate_inventory.csv")
summary = read_json("adjudicated_summary.json")
issues = read_csv("adjudicated_issue_families.csv")
positives = read_csv("positive_checks.csv")
cases = read_csv("adjudicated_cases.csv")
url_results = read_jsonl("url_results.jsonl")
pdf_results = read_jsonl("pdf_results.jsonl")
live_amac = read_csv("live_amac_comparison.csv")

assert len(inventory) == 825
assert len(url_results) == 1008 and all(200 <= row.get("status_code", 0) < 300 for row in url_results)
assert len(pdf_results) == 291 and all(not row.get("error") for row in pdf_results)
assert sum(row["match_basis"] == "none" for row in live_amac) == 1
print("审计产物与关键不变量校验通过。")

审计产物与关键不变量校验通过。


## Context & Methods

目标是复核私募基金合规语料的标题—正文—原始链接—分类—时效—附件关系。方法包括：

1. 扫描 7,008 个 canonical JSON，按私募基金、登记备案、私募资管、股权/创投基金和基金法等口径构造毛池。
2. 人工排除“向不特定合格投资者公开发行”、中小企业私募债和机构间私募产品/ABS 平台等 37 条词面误命中。
3. 隔离执行协会官网全分页抓取（2,525/2,525，失败 0），按相同口径取得 598 条官网私募候选并与 D 盘对账。
4. 访问全部 1,008 个官方来源 URL，检查状态、标题、日期、正文可比性和附件链接。
5. 检查 291 个本地 PDF 的结构、页数、文本层和 SHA；检查 358 个在线 PDF 签名及 18 个直接附件端点。
6. 对自动告警进行人工降噪：区分确认错误、包装页/PDF 来源不对称和校验限制；跨年度目视检查 8 个扫描 PDF 首页。

In [2]:
scope = summary["scope"]
rows = [
    ("全部 canonical", 7008),
    ("关键词毛池", scope["gross_keyword_candidates"]),
    ("排除误命中", scope["scope_exclusions"]),
    ("最终私募合规全集", scope["final_private_compliance_population"]),
    ("规则类", scope["rule_lane"]),
    ("参考/执法类", scope["reference_lane"]),
]
display(Markdown("\n".join(["|口径|数量|", "|---|---:|"] + [f"|{k}|{v:,}|" for k, v in rows])))

|口径|数量|
|---|---:|
|全部 canonical|7,008|
|关键词毛池|862|
|排除误命中|37|
|最终私募合规全集|825|
|规则类|146|
|参考/执法类|679|

## Data

In [3]:
domain_counts = Counter(row["url"].split("/")[2] for row in url_results)
effectiveness = Counter(row["effectiveness"] for row in inventory)
content_status = Counter(row["content_status"] for row in inventory)

display(Markdown("### 官方来源域名\n" + "\n".join(
    ["|域名|URL 数|", "|---|---:|"] + [f"|{k}|{v:,}|" for k, v in domain_counts.most_common()]
)))
display(Markdown("### 效力状态\n" + "\n".join(
    ["|状态|记录数|", "|---|---:|"] + [f"|{k}|{v:,}|" for k, v in effectiveness.most_common()]
)))
display(Markdown("### 内容状态\n" + "\n".join(
    ["|状态|记录数|", "|---|---:|"] + [f"|{k}|{v:,}|" for k, v in content_status.most_common()]
)))

### 官方来源域名
|域名|URL 数|
|---|---:|
|www.amac.org.cn|741|
|fg.amac.org.cn|152|
|neris.csrc.gov.cn|76|
|www.csrc.gov.cn|31|
|www.gzw.sh.gov.cn|5|
|jrj.sh.gov.cn|2|
|jrj.beijing.gov.cn|1|

### 效力状态
|状态|记录数|
|---|---:|
|not_applicable|679|
|current|76|
|unknown|56|
|historical|10|
|pending|4|

### 内容状态
|状态|记录数|
|---|---:|
|full_text|699|
|metadata_only|126|

## Results

In [4]:
positive_lines = ["|检查项|结果|通过/总数|说明|", "|---|---|---:|---|"]
for row in positives:
    positive_lines.append(
        f"|{row['check']}|{row['result']}|{int(row['numerator']):,}/{int(row['denominator']):,}|{row['note']}|"
    )
display(Markdown("### 正向校验\n" + "\n".join(positive_lines)))

### 正向校验
|检查项|结果|通过/总数|说明|
|---|---|---:|---|
|official_source_reachability|pass|1,008/1,008|all 1,008 official source URLs returned 2xx during the corrected verification run|
|live_amac_exact_url_and_title_match|partial|597/598|all 597 matched pages had exact normalized titles; one official page was missing|
|live_amac_text_coverage_at_least_80pct|pass_with_exceptions|448/465|17 low-coverage alerts were manually reviewed; most are wrapper/PDF-source asymmetry, with a confirmed incomplete duplicate wrapper|
|local_pdf_validity|pass|291/291|all PDFs parse; no recorded SHA mismatch|
|live_pdf_signature|pass|358/358|all live PDF source URLs begin with a valid PDF signature|
|nonempty_remote_attachment_hash_match|pass|13/13|every non-empty attachment endpoint with a local copy matched its expected SHA; five other endpoints are empty|
|manual_scan_pdf_title_and_document_match|pass|8/8|first pages sampled across 2017 and 2020-2026 visually matched the canonical title/entity and document type|

In [5]:
issue_lines = ["|优先级|严重度|问题族|影响|证据|", "|---:|---|---|---:|---|"]
for row in issues:
    issue_lines.append(
        f"|P{row['priority']}|{row['severity']}|{row['issue_family']}|{int(row['affected']):,} {row['unit']}|{row['evidence']}|"
    )
display(Markdown("### 人工裁决后的问题族\n\n> 各问题族有重叠，影响数不可相加。\n\n" + "\n".join(issue_lines)))

### 人工裁决后的问题族

> 各问题族有重叠，影响数不可相加。

|优先级|严重度|问题族|影响|证据|
|---:|---|---|---:|---|
|P0|critical|2026_disclosure_templates_have_unknown_effectivity|6 canonical records|six official disclosure templates published 2026-06-05 are unknown although the announcement sets 2026-09-01 implementation|
|P0|critical|future_repeal_of_2016_disclosure_rules_not_encoded|5 canonical records|official 2026 implementation rules repeal the 2016 management rule and format guides on 2026-09-01, but no successor chain is recorded|
|P0|critical|official_rule_page_missing_from_corpus|1 official page|AMAC live list page for 私募基金登记备案相关问题解答（十二） has no raw/canonical URL match|
|P0|critical|order_233_split_record_and_wrong_dates|3 canonical records|official rule page, NERIS full text, and AMAC news wrapper are split; NERIS dates are one day early and CSRC wrapper has only 140 characters|
|P0|critical|revised_2025_filing_guide_not_activated|2 canonical records|2025 revision is unknown with no dates while the superseded 2023 guide remains current|
|P1|high|malformed_publication_dates|17 canonical records|publication dates contain partial values such as 2026年1, 2026年3, or 2026年5|
|P1|high|obvious_news_or_one_off_material_in_rule_lane|11 canonical records|news wrappers, meeting coverage, or one-off institution notices are classified as rules|
|P1|high|official_attachment_endpoint_returns_empty_body|5 attachment URLs across 4 records|live NERIS endpoints return HTTP 200 with zero bytes and local download status is failed|
|P1|high|rule_lane_effectivity_unknown|56 rule-lane records|rule lane has no current/pending/historical decision|
|P1|high|rule_record_body_too_short|9 canonical records|rule-lane normalized body is under 200 characters, including zero-text attachments and CSRC wrappers|
|P1|high|same_official_url_split_into_duplicate_canonicals|24 records in 12 groups|each group contains multiple canonical IDs pointing to the same official URL|
|P1|high|systematic_neris_one_day_date_shift|76 canonical records|76 records have canonical publication date exactly one day before source UTC date|
|P2|medium|official_policy_subdomain_tls_chain_not_validated|152 official URLs|fg.amac.org.cn required TLS verification bypass in this runtime|
|P2|medium|scan_pdf_without_searchable_text|119 canonical records|131 records have scan-like PDFs; 119 remain metadata-only|

In [6]:
critical = [row for row in cases if row["severity"] == "critical"]
case_lines = ["|问题族|ID|标题|本地状态证据|", "|---|---|---|---|"]
for row in critical:
    case_lines.append(f"|{row['issue_family']}|{row['id'] or '官网缺失项'}|{row['title']}|{row['evidence']}|")
display(Markdown("### 关键问题明细\n" + "\n".join(case_lines)))

### 关键问题明细
|问题族|ID|标题|本地状态证据|
|---|---|---|---|
|2026_disclosure_templates_have_unknown_effectivity|law_040dd38ea0d0beabd76ad255|重要内容模板5号-私募证券投资基金信息披露清算报告|pub=2026-06-05 effective=missing status=unknown|
|2026_disclosure_templates_have_unknown_effectivity|law_2c8b6342b03c7e1d1dc370be|重要内容模板6号-私募股权、创业投资基金信息披露清算报告|pub=2026-06-05 effective=missing status=unknown|
|2026_disclosure_templates_have_unknown_effectivity|law_37b49bc189e778314b6d5215|重要内容模板2号-私募证券投资基金信息披露年度报告|pub=2026-06-05 effective=missing status=unknown|
|2026_disclosure_templates_have_unknown_effectivity|law_88269a674007f4a18f801771|重要内容模板1号-私募证券投资基金信息披露季度报告|pub=2026-06-05 effective=missing status=unknown|
|2026_disclosure_templates_have_unknown_effectivity|law_c7a0f09a43e65ab400a3346d|重要内容模板4号-私募股权、创业投资基金信息披露年度报告|pub=2026-06-05 effective=missing status=unknown|
|2026_disclosure_templates_have_unknown_effectivity|law_f0bbbe45511b0126561e7876|重要内容模板3号-私募股权投资基金信息披露半年度报告|pub=2026-06-05 effective=missing status=unknown|
|future_repeal_of_2016_disclosure_rules_not_encoded|law_17e0c98b1445597a59e6d7f5|私募投资基金信息披露内容与格式指引1号|pub=2016-02-04 effective=missing status=unknown|
|future_repeal_of_2016_disclosure_rules_not_encoded|law_2b8c4ae5b91882bb21e5e32d|私募投资基金信息披露内容与格式指引2号|pub=2016-11-14 effective=missing status=unknown|
|future_repeal_of_2016_disclosure_rules_not_encoded|law_350bb0a1cb8d08a2ee3cb0ba|私募投资基金信息披露管理办法|pub=2016-02-03 effective=2016-02-03 status=current|
|future_repeal_of_2016_disclosure_rules_not_encoded|law_3df59b520df1b11e1e5c8a68|私募投资基金信息披露内容与格式指引1号|pub=2015-11-26 effective=missing status=unknown|
|future_repeal_of_2016_disclosure_rules_not_encoded|law_666f5f3bc6eb6c79b0636063|私募投资基金信息披露内容与格式指引2号-适用于私募股权（含创业）投资基金|pub=2016-11-14 effective=missing status=unknown|
|official_rule_page_missing_from_corpus|官网缺失项|私募基金登记备案相关问题解答（十二）|live AMAC page had no exact URL or normalized-title match in canonical population|
|order_233_split_record_and_wrong_dates|law_080c4461cb2b7d347610c02a|私募投资基金信息披露监督管理办法|pub=2026-02-23 effective=2026-08-31 status=pending|
|order_233_split_record_and_wrong_dates|law_c1c4f9374cabdafb7a2be606|中国证监会发布《私募投资基金信息披露监督管理办法》|pub=2026-02-27 effective=2026-09-01 status=pending|
|order_233_split_record_and_wrong_dates|law_fa9879afe3753e34193e6901|【第233号令】《私募投资基金信息披露监督管理办法》|pub=2026-02-24 15:22:54 effective=2026-09-01 status=pending|
|revised_2025_filing_guide_not_activated|law_6b082e4ce08248b155df3d6a|私募投资基金备案指引第3号——私募投资基金变更管理人|pub=2023-09-27 effective=2023-09-27 status=current|
|revised_2025_filing_guide_not_activated|law_f8194c71817b4463d82a528d|私募投资基金备案指引第3号——私募投资基金变更管理人（2025年修订）|pub=missing effective=missing status=unknown|

In [7]:
scans = [row for row in pdf_results if row.get("scan_likely")]
pdf_stats = [
    ("本地 PDF", len(pdf_results)),
    ("有效 PDF", sum(not row.get("error") for row in pdf_results)),
    ("SHA 不一致", sum(row.get("sha_match") is False for row in pdf_results)),
    ("疑似扫描 PDF", len(scans)),
    ("扫描 PDF 涉及记录", len({rid for row in scans for rid in row.get("record_ids", [])})),
]
display(Markdown("### PDF 与扫描件\n" + "\n".join(
    ["|指标|数量|", "|---|---:|"] + [f"|{k}|{v:,}|" for k, v in pdf_stats]
)))

### PDF 与扫描件
|指标|数量|
|---|---:|
|本地 PDF|291|
|有效 PDF|291|
|SHA 不一致|0|
|疑似扫描 PDF|131|
|扫描 PDF 涉及记录|131|

## Takeaways

1. 标题与链接整体可信：协会官网独立清单中，已覆盖的 597 条均为精确 URL 和标准化标题匹配；没有发现成批标题错配。
2. 时效/效力是首要风险：NERIS 日期系统性提前一天，2025/2026 新规则与旧规则的 pending/current/historical 链条未正确表达。
3. 内容风险集中在结构性边角：同一官网 URL 被拆成多个 canonical ID、短包装页没有并入 PDF 正文、扫描执法材料缺少 OCR。
4. 附件真伪总体可靠：非空远程附件与本地 SHA 全部一致；5 个 NERIS 端点虽然 HTTP 200，但正文为空，应明确标记不可用并寻找替代官方副本。
5. 修复顺序应为：关键效力链与日期 → 漏页/重复 canonical → 规则/参考分类 → 扫描件 OCR → TLS 与附件可观测性。

局限：本次是 2026-07-16 的官网快照；`fg.amac.org.cn` 的 TLS 证书链在当前运行时无法验证，内容校验采用了明确记录的主机级绕过；自动正文相似度不能直接比较“HTML 包装页”与“PDF 正文”，因此相关低覆盖告警经过人工裁决后才纳入结论。